In [5]:
pip install TDCRPy==2.20.8

   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
   - -------------------------------------- 1.0/23.5 MB 11.9 MB/s eta 0:00:02
   --- ------------------------------------ 2.1/23.5 MB 8.4 MB/s eta 0:00:03
   ----- ---------------------------------- 3.1/23.5 MB 7.6 MB/s eta 0:00:03
   ----- ---------------------------------- 3.1/23.5 MB 7.6 MB/s eta 0:00:03
   ------- -------------------------------- 4.2/23.5 MB 4.7 MB/s eta 0:00:05
   --------- ------------------------------ 5.8/23.5 MB 5.0 MB/s eta 0:00:04
   ------------ --------------------------- 7.3/23.5 MB 5.6 MB/s eta 0:00:03
   --------------- ------------------------ 8.9/23.5 MB 5.7 MB/s eta 0:00:03
   ----------------- ---------------------- 10.5/23.5 MB 6.0 MB/s eta 0:00:03
   ------------------- -------------------- 11.5/23.5 MB 6.0 MB/s eta 0:00:02
   --------------------- ------------------ 12.8/23.5 MB 6.0 MB/s eta 0:00:02
   ------------------------- -------------- 14.7/23.5 MB 6.2 MB/s eta 0:00:02
 

In [6]:
import tdcrpy as td
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

In [7]:
Rad="Co-60"                   # radionuclides
pmf_1="1"                   # relatives fractions of the radionulides
kB =1.0e-5                  # Birks constant in cm keV-1
V = 10                      # volume of scintillator in mL
ne = 1000                   # number of discretization steps for the numerical computation of the quenching function
nq = td.TDCR_model_lib.readParameters()[14][0] # get the quantum efficiency

## Analytical model (precise but less accurated for complex decay schemes)

In [8]:
TD = 0.977667386529166     # TDCR parameter
TAB = 0.992232838598821    # TDCR parameter T/AB
TBC = 0.992343419459002    # TDCR parameter T/BC
TAC = 0.99275350064608     # TDCR parameter T/AC
minL = 0.1                 # minimum light yield (keV-1)
maxL = 20                  # maximum light yield (keV-1)

result1 = td.TDCRPy.effA(TD, Rad, pmf_1, kB, V, Lbounds=[minL, maxL])
print(f"\nResults using a global parameter")
print(f"Global free parameter: {result1[0]/nq} photons/keV")
print(f"Efficiency of single events: {result1[2]}")
print(f"Efficiency of double coincidence events: {result1[3]}")
print(f"Efficiency of triple coincidence events: {result1[4]}")

result3 = td.TDCRPy.effA([TD, TAB, TBC, TAC], Rad, pmf_1, kB, V, Lbounds=[minL, maxL])
print(f"\nResults using 3 free parameters")
print(f"Global free parameter: {result3[0]/nq} photons/keV")
print(f"Free parameter PMT A: {result3[1][0]/nq} photons/keV")
print(f"Free parameter PMT B: {result3[1][1]/nq} photons/keV")
print(f"Free parameter PMT C: {result3[1][2]/nq} photons/keV")
print(f"Efficiency of single events: {result3[2]}")
print(f"Efficiency of double coincidence events: {result3[3]}")
print(f"Efficiency of triple coincidence events: {result3[4]}")


Results using a global parameter
Global free parameter: 12.226664346360298 photons/keV
Efficiency of single events: 0.9692636202453446
Efficiency of double coincidence events: 0.9722502553725052
Efficiency of triple coincidence events: 0.9505373614303487

Results using 3 free parameters
Global free parameter: 12.226664346360298 photons/keV
Free parameter PMT A: 12.162054737364487 photons/keV
Free parameter PMT B: 12.497405986769015 photons/keV
Free parameter PMT C: 12.084901810257893 photons/keV
Efficiency of single events: 0.9505916691144345
Efficiency of double coincidence events: 0.9722821827128985
Efficiency of triple coincidence events: 0.9505916691144345


## Stochastic model (less precise but aimed to be more accurate for complex decay schemes)

In [9]:
N = 10000 # number of Monte-Carlo trials
result2 = td.TDCRPy.eff(TD, Rad, pmf_1, kB, V, N, Lbounds=(minL, maxL))

In [10]:
print(f"global free parameter = {round(result2[0],4)} photons/keV")
print(f"global free parameter (PMT A) = {round(result2[1][0],4)} photons/keV")
print(f"global free parameter (PMT B) = {round(result2[1][1],4)} photons/keV")
print(f"global free parameter (PMT C) = {round(result2[1][2],4)} photons/keV")
print(f"efficiency S = {round(result2[2],4)} +/- {round(result2[3],4)}")
print(f"efficiency D = {round(result2[4],4)} +/- {round(result2[5],4)}")
print(f"efficiency T = {round(result2[6],4)} +/- {round(result2[7],4)}")
print(f"efficiency AB = {round(result2[6],4)} +/- {round(result2[7],4)}")
print(f"efficiency BC = {round(result2[8],4)} +/- {round(result2[9],4)}")
print(f"efficiency AC = {round(result2[10],4)} +/- {round(result2[11],4)}")

global free parameter = 9.9991 photons/keV
global free parameter (PMT A) = 9.9991 photons/keV
global free parameter (PMT B) = 9.9991 photons/keV
global free parameter (PMT C) = 9.9991 photons/keV
efficiency S = 0.9863 +/- 0.0009
efficiency D = 0.9736 +/- 0.0013
efficiency T = 0.951 +/- 0.0018
efficiency AB = 0.951 +/- 0.0018
efficiency BC = 0.9585 +/- 0.0016
efficiency AC = 0.9585 +/- 0.0016


## Comparison

In [11]:
print(f"Efficiency of double coincidence events (deviation): {result1[3]-result2[4]}")
print(f"Efficiency of triple coincidence events (deviation): {result1[4]-result2[6]}")

Efficiency of double coincidence events (deviation): -0.00131978884274786
Efficiency of triple coincidence events (deviation): -0.000466491730646168
